In [ ]:
# JupyterLite 环境准备：安装本题依赖，并按需下载本题素材
from pathlib import Path
import asyncio
import hashlib
import piplite
from pyodide.http import pyfetch

PACKAGES = ["pandas", "scikit-learn", "openpyxl"]
ASSETS = [{"name": "大学生低碳生活行为的影响因素数据集.xlsx", "url": "/labs/2.1.2/%E5%A4%A7%E5%AD%A6%E7%94%9F%E4%BD%8E%E7%A2%B3%E7%94%9F%E6%B4%BB%E8%A1%8C%E4%B8%BA%E7%9A%84%E5%BD%B1%E5%93%8D%E5%9B%A0%E7%B4%A0%E6%95%B0%E6%8D%AE%E9%9B%86.xlsx", "size": 63152, "sha256": "aa3ccf118a49b8b5492b8701b374d8eb5afe516ae10d0a164c41371107120322"}]

async def _retry(label, operation, attempts=3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return await operation()
        except Exception as error:
            last_error = error
            if attempt < attempts:
                await asyncio.sleep(0.8 * attempt)
    raise RuntimeError(f"{label}失败（已重试 {attempts} 次）：{last_error}") from last_error

if PACKAGES:
    await _retry("依赖安装", lambda: piplite.install(PACKAGES))

for asset in ASSETS:
    target = Path(asset["name"])
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() == asset["sha256"]:
        continue
    async def _download():
        separator = "&" if "?" in asset["url"] else "?"
        response = await pyfetch(f'{asset["url"]}{separator}sha256={asset["sha256"][:16]}')
        if response.status < 200 or response.status >= 300:
            raise RuntimeError(f'HTTP {response.status}：{asset["url"]}')
        return await response.bytes()
    payload = await _retry(f'素材下载 {asset["name"]}', _download)
    digest = hashlib.sha256(payload).hexdigest()
    if digest != asset["sha256"]:
        raise RuntimeError(f"素材校验失败：{asset['name']}")
    target.write_bytes(payload)

print("✓ 2.1.2 环境已就绪", f"依赖 {len(PACKAGES)} 项，素材 {len(ASSETS)} 个")


In [ ]:
import pandas as pd
#读取一个Excel文件，并将读取到的数据存储在变量data中
data = __________
#打印出数据集的前5行
print(data.head())

#处理数据集中的缺失值
initial_row_count = __________   #处理前的数据行数
data = __________                #删除缺失值所在行
final_row_count = __________     #处理后的数据行数
print(f'处理后数据行数: {final_row_count}, 删除的行数: {initial_row_count - final_row_count}')

#删除重复行
data = __________

from sklearn.preprocessing import StandardScaler
numerical_features = ['4.您的月生活费○≦1,000元   ○1,001-2,000元   ○2,001-3,000元   ○≧3,001元']
scaler = StandardScaler()
data[numerical_features] = __________

#选择特征
selected_features = [__________]
X = __________

# 创建目标变量
y = __________

from sklearn.model_selection import train_test_split
# 数据划分（测试集取20%）
X_train, X_test, y_train, y_test = __________(__________, random_state=42)

# 合并处理后得数据，并将其保存（保存中不用额外创建索引）
cleaned_data = __________(__________, axis=1)
__________('2.1.2_cleaned_data.csv', __________)